In [ ]:
import ollama

_instalados = [m["model"] for m in ollama.list()["models"]] #modelos instalados
MODELO_EMBED = next((t for t in ["embddinggemma", "embeddinggemma:300m", "embeddinggemma:latest"] if t in _instalados), "embeddinggemma") 

MODELO_CHAT = "llama3:latest" #el modelo que redacta las respuestas, se puede cambiar por otro modelo de ollama

print("Embedding:", MODELO_EMBED)
print("Chat:", MODELO_CHAT)
print("Instalados:", MODELO_EMBED in _instalados, MODELO_CHAT in _instalados)

In [ ]:
def preguntar(prompt, system=None):
    """Manda un mensaje al modelo de CHAT y devuelve solo el texto de la respuesta.
    prompt :  la pregunta o instrucción del usuario (rol "user").
    system : instrucciones de comportamiento opcionales para el modelo (rol "system").
        Aquí añadiremos más adelante las reglas del RAG.
    """

    # ollama espera una LISTA de mensajes con su rol (igual que la API de OPENAI)
    mensajes = []
    if system:
        mensajes.append({"role": "system", "content": system})
    mensajes.append({"role": "user", "content": prompt})

    opciones = {"temperature": 0} #opciones de generación, se pueden cambiar
    if "qwen" in MODELO_CHAT:
        opciones["think"] = False #qwen3 tiene una opción para "pensar" antes de responder, pero no siempre da buenos resultados

    r = ollama.chat(model=MODELO_CHAT, messages=mensajes, options=opciones)
    return r["message"]["content"].strip() #nos quedamos solo con el texto sin metadatos

print(preguntar("Di 'listo'"))

In [ ]:
pregunta_de_prueba = "¿Cuántos días de vacaciones me corresponden al año en Lumetra?"

print(preguntar(pregunta_de_prueba))

In [ ]:
r = ollama.embed(model=MODELO_EMBED, input="El gato duerme en el sofá")
vector = r["embeddings"][0] #el resultado es una lista de listas, nos quedamos con la primera

print(f"Dimensión del vector: {len(vector)}") 
print(f"Primeras 5 componentes: {vector[:5]}")

In [ ]:
import numpy as np

def coseno(a, b):
    """Calcula la similitud coseno entre dos vectores a y b."""
    a = np.array(a)
    b = np.array(b)
    # producto escalar (a @ b) dividido por el producto de las normas
    return float(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

frases = [
    "El gato duerme en el sofá",
    "Un felino descansa en el sillón",
    "La factura vence el 30 de junio",
    "El pago debe realizarse antes de fin de mes"
]

vecs = ollama.embed(model=MODELO_EMBED, input=frases)["embeddings"] #vectorizamos todas las frases


# comparamos todas las parejas posibles de frases
for i in range(len(frases)):
    for j in range(i+1, len(frases)):       
        print(f"{coseno(vecs[i], vecs[j]):.3f}  |  '{frases[i]}'  <->  '{frases[j]}'")
 

In [ ]:
# Embeddinggemma se entrenó con "prefijos" que le indican qué papel juega el texto.
# Usar el prefijoi correcto mejora el emparejamiento pregunta <-> documento.

def embed_documentos(textos):
    """Vectoriza una LISTA de textos que vamos a guardar en la base (documentos)."""
    entrada = [f"title: none | text: {t}" for t in textos] #añadimos el prefijo (de documentos) "title: none | text:" a cada texto
    return ollama.embed(model=MODELO_EMBED, input=entrada)["embeddings"]

def embed_consulta(texto):
    """Vectoriza UNA pregunta que vamos a BUSCAR (consulta)."""
    entrada = f"task: search result | query: {texto}" #añadimos el prefijo (de consulta) "query:" a la pregunta
    return ollama.embed(model=MODELO_EMBED, input=entrada)["embeddings"][0] #devuelve un solo vector

 

In [ ]:
comentarios = [
    "Me han cobrado dos veces la suscripción este mes",
    "La app va lentísima desde la última actualización",
    "No consigo restablecer mi contraseña, el correo no llega",
    "El cargo de mi tarjeta no corresponde con la tarifa contratada",
    "Me encanta el nuevo diseño del panel de control, ¡muy intuitivo!",
    "Quiero darme de baja y nadie me responde"
]

def buscar (consulta, textos, k=3):
    """Busca los k textos más relevantes para la consulta dada, usando similitud coseno.
    Es un TAG en miniatura: vectorizar todo -> comparar -> ordenar -> quedarnos con los mejores.    
    """
    vec_consulta = embed_consulta(consulta) #vectorizamos la consulta
    vecs_textos = embed_documentos(textos) #vectorizamos los textos (documentos)

    #calculamos la similitud coseno entre la consulta y cada texto
    puntuaciones = [coseno(vec_consulta, v) for v in vecs_textos]

    orden = sorted(range(len(textos)), key=lambda i: puntuaciones[i], reverse=True) #ordenamos los índices de los textos por su puntuación
    return [(textos[i], puntuaciones[i]) for i in orden[:k]] #devolvemos los k mejores textos con su puntuación 


for texto, p in buscar("problemas para pagar", comentarios):
    print(f"{p:.3f}  |  {texto}")
 

In [ ]:
from pathlib import Path

# Cargar todos los .txt de la carpeta datos/ en un diccionario {nombre: contenido}
docs = {}
for ruta in sorted(Path("datos").glob("*.txt")):
    docs[ruta.name] = ruta.read_text(encoding="utf-8")

for nombre, texto in docs.items():
    print(f"{nombre}: {len(texto)} caracteres")    

In [ ]:
# Troceamos cada documento en CHUNKS

chunks, metadatos = [], []
for nombre, texto in docs.items():
    for parrafo in texto.split("\n\n"): #los párrados por una línea en blanco
        parrafo = parrafo.strip() #quitamos espacios al principio y al final
        if len(parrafo) > 40: #si el párrafo tiene más de 40 caracteres, lo guardamos como chunk
            chunks.append(parrafo)
            metadatos.append({"fuente": nombre}) #guardamos el nombre del documento como metadato


print(f"{len(chunks)} chunks generados")   
print("Ejemplo de chunk:", chunks[0][:120],"...")         

In [ ]:
import chromadb

# PersistentClient: los datos se guardan en disco y persisten entre ejecuciones
cliente = chromadb.PersistentClient(path="chroma_lumetra") 

# Una "colección" es como una tabla de base de datos donde guardaremos nuestros chunks y sus vectores
coleccion =  cliente.get_or_create_collection(
    "lumetras", metadata={"hnsw:space": "cosine"}
)


# upsert: inserta o actualiza un vector con su id, sus metadatos y su chunk de texto original
# Guardamos 4 cosas por chunk: id, texto, vector y metadato

coleccion.upsert(
    ids=[f"chunk_{i}" for i in range(len(chunks))], #id único para cada chunk
    documents=chunks, #el texto original del chunk
    embeddings=embed_documentos(chunks), #el vector del chunk
    metadatas=metadatos #los metadatos (en este caso, la fuente)
)

print(f"Chunks guardados en la colección: {coleccion.count()}")

In [ ]:
def recuperar(pregunta, k=4):
    """Función de recuperación que busca los k chunks más relevantes para la pregunta dada.

     Usamos k=4 (y no 1) porque el chunk con el dato exacto no siempre es el #1 en la lista de resultados,
     a veces el modelo lo considera relevante pero no el más relevante.
     De esta forma, aunque el chunk con la respuesta exacta no sea el #1,
     al menos estará entre los 4 primeros y el modelo de CHAT podrá usarlo para redactar la respuesta.

    """
    #query_embeddings espera una lista de vectores de consulta, pero nosotros solo tenemos una pregunta, así que le pasamos una lista con un solo vector
    res = coleccion.query(query_embeddings=[embed_consulta(pregunta)], n_results=k)   
    #chroma devuelve listas de listas, nos quedamos con la primera (y única) consulta y hacemos zip para juntar cada chunk con su fuente 
    return list(zip(res["documents"][0], [m["fuente"] for m in res["metadatas"][0]])) 

for texto, fuente in recuperar("¿Cuántos días de vacaciones me corresponden?"):
    print(f"{fuente}: {texto[:110]}...")

In [ ]:
for texto, fuente in recuperar("¿me pagan algo por currar desde casa?"):
    print(f"{fuente}: {texto[:110]}...")

Ejercicio

1. Probad 3 preguntas sobre Lumetra, ¿ sale el chunk correcto en la respuesta?
2. Construir una pregunta con sinonimos
3. Intented encontrar una pregunta donde el top 1 de chunk que sale este equivocado

In [ ]:
# 1. Probad 3 preguntas sobre Lumetra, ¿ sale el chunk correcto en la respuesta?
preguntas = [
    "¿Qué es Lumetra?",
    "¿Qué servicios ofrece Lumetra?",
    "¿Cuál es el objetivo principal de Lumetra?"
]

for pregunta in preguntas:
    print("\n" + "="*80)
    print("PREGUNTA:", pregunta)

    resultados = recuperar(pregunta)

    for i, (texto, fuente) in enumerate(resultados, start=1):
        print(f"\nChunk {i}")
        print(f"Fuente: {fuente}")
        print(texto[:500])

In [ ]:
# 2. Pregunta con sinónimos
for texto, fuente in recuperar(
    "¿Qué soluciones proporciona Lumetra a sus clientes?"
):
    print(f"{fuente}: {texto[:300]}")

In [ ]:
# 3. Intented encontrar una pregunta donde el top 1 de chunk que sale este equivocado
preguntas_dificiles = [
    "¿Qué ventajas tiene?",
    "¿Cómo funciona?",
    "¿Qué ofrece la empresa?",
    "¿Me pagan algo por trabajar desde casa?"
]

for pregunta in preguntas_dificiles:
    print("\n" + "="*80)
    print("PREGUNTA:", pregunta)

    for texto, fuente in recuperar(pregunta):
        print(f"{fuente}: {texto[:200]}")

In [ ]:
SYSTEM_RAG ="""Eres el asistente interno de Lumetra.
Responde SOLO con la información del CONTEXTO.
Si la respuesta no está en el contexto, di exactamente: "No encuentro esa información en la documentación".
Cita siempre el documento del que sacas cada dato, entre corchetes:  [nombre_del_archivo]"""

def rag(pregunta, k=4, ver_contexto=False):
    """RAG completo = recuperar + generar respuesta."""

    trozos = recuperar(pregunta, k) #recuperamos los k chunks más relevantes para la pregunta

    contexto = "\n\n".join([f"[{fuente}]: {texto}" for texto, fuente in trozos]) #creamos un contexto con los chunks recuperados, indicando su fuente entre corchetes

    if ver_contexto:
        print("-" * 60, f"\n CONTEXTO RECUPERADO: \n\n{contexto}\n", "-" * 60)

    prompt = f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"
    return preguntar(prompt, system=SYSTEM_RAG) #mandamos el prompt al modelo de CHAT con las instrucciones del sistema    

 

In [ ]:
# comparamos la MISMA pregunta con y sin RAG
print("--- SIN RAG ---")
print(preguntar(pregunta_de_prueba))
print("\n--- CON RAG ---")
print(rag(pregunta_de_prueba))

In [ ]:
# comparamos la MISMA pregunta con y sin RAG
print("--- SIN RAG ---")
print(preguntar(pregunta_de_prueba))
print("\n--- CON RAG ---")
print(rag("¿Cuántos días de vacaciones me corresponden al año en Lumetra?", ver_contexto=True))

Probad el rag()
* 3 Preguntas con respuesta -> presente en los documentos
* 1 pregunta sin respuesta
* 1 pregunta que cruce dos documentos
* ¿ Siempre cita las fuentes?


In [ ]:
# Preguntas hechas por mi
preguntas = [
    "¿Qué es Lumetra?",
    "¿Qué servicios ofrece Lumetra?",
    "¿Qué problema resuelve Lumetra para sus clientes?"
]

for pregunta in preguntas:
    print("="*80)
    print("PREGUNTA:", pregunta)
    print()
    
    respuesta = rag(pregunta)
    
    print("RESPUESTA:")
    print(respuesta)

In [ ]:
# SOLUCIÓN del profesor (batería de ejemplo) — preguntas variadas para estresar al asistente:
baterias = [
    "¿Qué días tengo que ir a la oficina obligatoriamente?",
    "¿Qué incluye el plan Pro y cuánto cuesta?",
    "¿Qué hago si no me llega el correo para resetear la contraseña?",
    "¿Cuál es la política de bajas por enfermedad?",   # ¡NO está en el corpus! → debe confesarlo
    "¿Puedo teletrabajar todo julio y cuántos días seguidos de vacaciones puedo coger en agosto?",  # cruza 2 docs
]
for p in baterias:
    print(f"\n❓ {p}")
    print(rag(p))

## Ejercicio:

**a) El efecto de `k`.** Lanza la misma pregunta con `k=1` y `k=8`. ¿Cuándo le falta información? ¿Cuándo le sobra ruido? (con `ver_contexto=True` se ve clarísimo).

**b) Tu propio documento.** Crea `datos/politica_formacion.txt` (invéntate la política: presupuesto anual, cómo se pide…). Re-ejecuta las celdas de chunking + indexado y pregúntale por ella. Acabas de "enseñarle" algo nuevo al sistema **sin tocar el modelo**.

**c) Mini-eval.** Pasa la batería de abajo con y sin RAG y cuenta aciertos. De esta forma puedes comparar modelos y configuraciones del RAG.

**d) (Si te sobra tiempo)** Cambia el chunking: trocea por tamaño fijo (p. ej. 300 caracteres) en vez de por párrafos y repite la mini-eval. ¿Mejora o empeora? ¿Por qué?
 

In [ ]:
preguntas_eval = [
    "¿Cuántos días de vacaciones anuales tengo?",
    "¿Qué días son obligatorios en oficina?",
    "¿Cuánto cuesta el plan Pro?",
    "¿En qué navegador falla la exportación a PDF?",
    "¿Qué curso es obligatorio la primera semana?",
    "¿Cuál es la política de bajas por enfermedad?",
]

In [ ]:
# Con RAG
for pregunta in preguntas_eval:
    print("=" * 80)
    print("PREGUNTA:", pregunta)
    
    respuesta = rag(pregunta, k=3, ver_contexto=True)
    
    print("RESPUESTA:")
    print(respuesta)

In [ ]:
# Con k=1 vs k=8
for pregunta in preguntas_eval:
    print("=" * 80)
    print("PREGUNTA:", pregunta)

    print("\n--- k=1 ---")
    print(rag(pregunta, k=1, ver_contexto=True))

    print("\n--- k=8 ---")
    print(rag(pregunta, k=8, ver_contexto=True))

### Ejercicio (b) — Tu propio documento

Re-indexamos todos los documentos (incluyendo `politica_formacion.txt` que ahora tiene contenido) con el pipeline completo: cargar → trocear → indexar. Luego preguntamos por el nuevo documento.

In [ ]:
from pathlib import Path
import chromadb

# Recargar todos los documentos .txt
docs = {}
for ruta in sorted(Path("datos").glob("*.txt")):
    docs[ruta.name] = ruta.read_text(encoding="utf-8")

# Trocear en chunks (por párrafos)
chunks, metadatos = [], []
for nombre, texto in docs.items():
    for parrafo in texto.split("\n\n"):
        parrafo = parrafo.strip()
        if len(parrafo) > 40:
            chunks.append(parrafo)
            metadatos.append({"fuente": nombre})

print(f"Total chunks: {len(chunks)}")
for m in set(m["fuente"] for m in metadatos):
    print(f"  - {m}: {sum(1 for x in metadatos if x['fuente'] == m)} chunks")

In [ ]:
# Indexar (o re-indexar) en ChromaDB
cliente = chromadb.PersistentClient(path="chroma_lumetra")
coleccion = cliente.get_or_create_collection("lumetras", metadata={"hnsw:space": "cosine"})

# upsert sobreescribe los vectores anteriores
coleccion.upsert(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=embed_documentos(chunks),
    metadatas=metadatos
)

print(f"Chunks indexados en ChromaDB: {coleccion.count()}")

In [ ]:
# Probar preguntas sobre el nuevo documento
print(rag("¿Cuánto presupuesto anual tengo para formación?"))
print()
print(rag("¿Cómo solicito un curso de formación?"))
print()
print(rag("¿Cuántos días de antelación necesito para pedir una formación?"))

### Ejercicio (d) — Chunking por tamaño fijo (300 caracteres)

Probamos otra estrategia de troceado: en vez de separar por párrafos, cortamos cada documento en fragmentos de 300 caracteres con solapamiento (overlap) de 50 caracteres. Luego comparamos resultados con la mini-eval.

In [ ]:
def chunk_fijo(texto, tamano=300, solapamiento=50):
    """Trocea un texto en fragmentos de tamaño fijo con solapamiento."""
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fin = min(inicio + tamano, len(texto))
        fragmento = texto[inicio:fin].strip()
        if len(fragmento) > 40:
            chunks.append(fragmento)
        inicio += tamano - solapamiento
    return chunks

# Cargar documentos
docs = {}
for ruta in sorted(Path("datos").glob("*.txt")):
    docs[ruta.name] = ruta.read_text(encoding="utf-8")

# Trocear por tamaño fijo
chunks_fijo, metadatos_fijo = [], []
for nombre, texto in docs.items():
    for fragmento in chunk_fijo(texto):
        chunks_fijo.append(fragmento)
        metadatos_fijo.append({"fuente": nombre})

print(f"Chunks (tamaño fijo): {len(chunks_fijo)}")
print(f"Chunks anteriores (párrafos): {len(chunks)}")
print()
print("Ejemplo de chunk fijo:", chunks_fijo[0][:120])

In [ ]:
# Crear una colección aparte para no mezclar con la de párrafos
coleccion_fijo = cliente.get_or_create_collection(
    "lumetras_fijo", metadata={"hnsw:space": "cosine"}
)

coleccion_fijo.upsert(
    ids=[f"chunk_fijo_{i}" for i in range(len(chunks_fijo))],
    documents=chunks_fijo,
    embeddings=embed_documentos(chunks_fijo),
    metadatas=metadatos_fijo
)

print(f"Chunks indexados (tamaño fijo): {coleccion_fijo.count()}")

In [ ]:
def recuperar_fijo(pregunta, k=4):
    """Recupera chunks de la colección de tamaño fijo."""
    res = coleccion_fijo.query(
        query_embeddings=[embed_consulta(pregunta)], n_results=k
    )
    return list(zip(res["documents"][0], [m["fuente"] for m in res["metadatas"][0]]))

def rag_fijo(pregunta, k=4, ver_contexto=False):
    """RAG con chunks de tamaño fijo."""
    trozos = recuperar_fijo(pregunta, k)
    contexto = "\n\n".join([f"[{fuente}]: {texto}" for texto, fuente in trozos])
    if ver_contexto:
        print("-" * 60, f"\n CONTEXTO (FIJO): \n\n{contexto}\n", "-" * 60)
    prompt = f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"
    return preguntar(prompt, system=SYSTEM_RAG)

In [ ]:
# Mini-eval comparativo: chunk por párrafos vs tamaño fijo
print(f"{'PREGUNTA':<55} {'PÁRRAFOS':<35} {'TAMAÑO FIJO':<35}")
print("=" * 125)

for pregunta in preguntas_eval:
    r1 = rag(pregunta, k=3)
    r2 = rag_fijo(pregunta, k=3)
    r1_corto = r1[:32] + "..." if len(r1) > 35 else r1
    r2_corto = r2[:32] + "..." if len(r2) > 35 else r2
    print(f"{pregunta:<55} {r1_corto:<35} {r2_corto:<35}")

**Preguntas para analizar:**
1. ¿Qué estrategia da respuestas más precisas?
2. El chunking fijo puede romper frases por la mitad → ¿el modelo de chat lo nota?
3. ¿Cuándo conviene usar párrafos y cuándo tamaño fijo?